# 03 — Benchmark Evaluation

Evaluates SatQuery AI specialist models across standard remote sensing benchmarks.

## Benchmarks
- **BigEarthNet**: Multi-label scene classification (R@1, R@5)
- **Change Detection**: Change estimation accuracy
- **VQA / Captioning**: Qualitative outputs on test images


In [ ]:
import sys
sys.path.insert(0, '../backend')

import os
import json
import numpy as np
import torch
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINT = os.environ.get('CLIP_CHECKPOINT', '../backend/checkpoints/clip_bigearthnet_best.pt')
DATA_DIR = os.environ.get('BIGEARTHNET_DIR', '/data/BigEarthNet')

print(f'Device: {DEVICE}')
print(f'Checkpoint exists: {os.path.exists(CHECKPOINT)}')

In [ ]:
# Load model registry
from config import get_settings
from models.registry import ModelRegistry

settings = get_settings()
registry = ModelRegistry.get_instance()
registry.load_all(settings)

for m in registry.list_available():
    status = 'OK' if m['loaded'] else 'FAILED'
    print(f'  [{status}] {m["name"]} — {m["task"]}')

In [ ]:
# CLIP Retrieval Evaluation
import open_clip
import torch.nn.functional as F
from torch.utils.data import DataLoader
from training.bigearthnet_dataset import BigEarthNetDataset, BIGEARTHNET_43_LABELS
from training.evaluate import CLIPRetrievalEvaluator

model, _, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
tokenizer = open_clip.get_tokenizer('ViT-B-32')

if os.path.exists(CHECKPOINT):
    ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded checkpoint (epoch={ckpt.get("epoch")}, R@1={ckpt.get("val_r1", 0):.2f}%)')

model = model.to(DEVICE)

test_ds = BigEarthNetDataset(DATA_DIR, split='test', use_sar=False, max_samples=300)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

evaluator = CLIPRetrievalEvaluator(model, tokenizer, DEVICE)
clip_results = evaluator.evaluate(test_loader, BIGEARTHNET_43_LABELS)

print('\n=== CLIP Retrieval ===')
for group, scores in clip_results.items():
    print(f'  {group}:')
    for k, v in scores.items():
        print(f'    {k}: {v:.2f}%')

In [ ]:
# Change Detection
from PIL import Image

change_model = registry.get('ChangeDetectionModel')

def make_test_pair(change_pct=0.2, size=256):
    rng = np.random.default_rng(42)
    base = rng.integers(0, 255, (size, size, 3), dtype=np.uint8)
    modified = base.copy()
    n_pixels = int(size * size * change_pct)
    rows = rng.integers(0, size, n_pixels)
    cols = rng.integers(0, size, n_pixels)
    modified[rows, cols] = rng.integers(0, 255, (n_pixels, 3), dtype=np.uint8)
    return Image.fromarray(base), Image.fromarray(modified)

print('=== Change Detection Results ===')
change_results = []
for true_pct in [0, 5, 15, 30, 60]:
    img1, img2 = make_test_pair(change_pct=true_pct / 100)
    out = change_model.detect_changes(img1, img2)
    detected = out['change_percentage']
    change_results.append({'true': true_pct, 'detected': detected})
    print(f'  True={true_pct:3d}%  Detected={detected:5.1f}%  Error={abs(true_pct - detected):5.1f}%')

In [ ]:
# VQA & Captioning samples
vqa_model = registry.get('RemoteSensingVQA')
cap_model = registry.get('RemoteSensingCaptioning')

rng = np.random.default_rng(0)
test_arr = rng.integers(50, 200, (256, 256, 3), dtype=np.uint8)
test_arr[64:128, 64:128] = [30, 150, 40]   # green patch
test_arr[128:192, 128:192] = [20, 60, 180]  # blue patch
test_img = Image.fromarray(test_arr)

questions = [
    'What do you see in this image?',
    'Is there vegetation?',
    'What land cover types are present?',
]

print('=== VQA Results ===')
for q in questions:
    ans = vqa_model.answer(test_img, q)
    print(f'  Q: {q}')
    print(f'  A: {ans["answer"]}  (conf={ans["confidence"]:.2f})')

caption = cap_model.generate_caption(test_img)
print(f'\nCaption: {caption["caption"]}  (conf={caption["confidence"]:.2f})')

In [ ]:
# Summary bar chart
summary = {
    'CLIP I→T R@1': clip_results['image_to_text'].get('R@1', 0),
    'CLIP I→T R@5': clip_results['image_to_text'].get('R@5', 0),
    'CLIP I→I R@1': clip_results['image_to_image'].get('R@1', 0),
    'Change MAE (%)': np.mean([abs(r['true'] - r['detected']) for r in change_results]),
}

fig, ax = plt.subplots(figsize=(9, 4))
fig.patch.set_facecolor('#0a1020')
ax.set_facecolor('#0f1a30')

bars = ax.bar(list(summary.keys()), list(summary.values()),
              color=['#3aabff', '#60c5ff', '#1e7fc8', '#f59e0b'], alpha=0.85)
ax.set_ylabel('Value (%)', color='white')
ax.set_title('SatQuery AI — Benchmark Summary', color='white', fontsize=12)
ax.tick_params(colors='white')
ax.spines[['top', 'right']].set_visible(False)
for spine in ax.spines.values():
    spine.set_color('#334155')

for bar, val in zip(bars, summary.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{val:.1f}', ha='center', color='white', fontsize=9)

plt.tight_layout()
plt.show()

with open('benchmark_results.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('\nResults saved to benchmark_results.json')